# DREAMwalk MSI: disease-area split (Figure 2b)

Compare the **original** and **combined modified** DREAMwalk embeddings on unseen disease categories. Select the existing DREAMwalk **Python 3.8 kernel**, then run the cells in order or choose **Run All**. Classifier training starts in Sections 6 and 7.

Interpreter relative to the repository: `00.Reproduction/00.DREAMwalk_reproduction/environment/.conda-env/bin/python`.

This notebook reuses two completed MeSH2024, seed-0, tau=0.3, pp=1 embeddings. Sections 1-10 retain the completed strict all-membership evaluation. Sections 11-16 add a fixed-primary-category evaluation with the same category allocations and pair IDs. Each policy trains 20 XGBoost classifiers: two variants on ten shared splits. Each completed fit is saved before the next fit.

If strict evaluation is already complete, run Sections 1-5, then Sections 11-16. New classifier training starts in Sections 13 and 14. Run All also works and reuses completed strict fits.

Disease nodes remain in the embedding graph. All DDA edges were already excluded from embedding generation. This experiment holds out **classifier labels**, so it measures generalization to disease areas among nodes with existing embeddings.

[Bang et al. (2023)](https://doi.org/10.1038/s41467-023-39301-y), Figure 2b and Methods (page 14), specify approximately 8:1:1 MeSH-category splits repeated ten times. The authors did not release category assignments or split IDs. The choices below define the strict reconstruction in Sections 1-10:

- Use every highest-level category from the accepted MeSH2024 mapping, such as `C04`, `C14`, or `F03`.
- A disease touching a test category goes entirely to test. Otherwise, one touching validation goes entirely to validation. All remaining diseases go to training.
- Exclude the 92 diseases without a usable category and their 330 positive pairs from classifier evaluation. Their graph nodes and embeddings are retained.
- Generate ten globally balanced negative sets with no cross-repeat overlap, then split each set by disease. Individual partitions need not have a 1:1 class ratio.
- Both variants use identical pairs and splits. XGBoost parameters and training seed remain fixed. Validation is reported without parameter selection or early stopping.
- Local SD describes split/negative-set variation conditional on the two fixed embeddings.


## 1. Paths and existing environment

Run this notebook in the existing DREAMwalk Python 3.8 environment. If it starts outside the repository, set `PROJECT_OVERRIDE` to the DREAMwalk project directory.


In [13]:
from pathlib import Path
import sys
import csv
import hashlib
import json
import platform
from importlib.metadata import version

PROJECT_OVERRIDE = None
start = Path(PROJECT_OVERRIDE).expanduser().resolve() if PROJECT_OVERRIDE else Path.cwd().resolve()
ROOT = next((candidate
             for parent in [start, *start.parents]
             for candidate in [parent, parent/'00.DREAMwalk_reproduction',
                               parent/'00.Reproduction/00.DREAMwalk_reproduction']
             if (candidate/'src/msi_disease_area_split.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Set PROJECT_OVERRIDE to the DREAMwalk project directory.')
if sys.version_info[:2] != (3, 8):
    raise RuntimeError('Select the existing DREAMwalk Python 3.8 kernel: ' +
                       str(ROOT/'environment/.conda-env/bin/python'))
sys.path.insert(0, str(ROOT/'src'))
import numpy as np
import pandas as pd
from IPython.display import display
import msi_disease_area_split as area
from run_msi_repeat0 import xgb_settings, sha256

print('Project:', ROOT)
print('Interpreter:', sys.executable)
print('Packages:', {name: version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'xgboost']})


Project: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction
Interpreter: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/environment/.conda-env/bin/python
Packages: {'numpy': '1.23.3', 'pandas': '1.4.4', 'scikit-learn': '1.1.3', 'xgboost': '1.7.4'}


## 2. Fixed comparison settings

The original variant retains the released teleport selector and input-type pp=1 NS. The modified variant uses first-hit teleport, positive-output-type NS, and current-positive rejection. Both use the same MeSH2024 semantic input at cutoff 0.4.

The repeat seed controls category splitting and Link-level negative sampling. The classifier training seed is 0 for both variants in every repeat. Accuracy uses threshold 0.5; AUPR is average precision.


In [14]:
RUN_NAME = 'msi_mesh2024_disease_area_seed0_9'
REPEAT_SEEDS = list(range(10))
RESULTS = ROOT/'results'/RUN_NAME
XGB_PARAMETERS = xgb_settings()
MODELS = {
    'original': 'msi_seed0_semantic_pp_mesh2024_roots',
    'modified': 'msi_seed0_mesh2024_final_fixed',
}
EMBEDDINGS = {name: ROOT/'data/embeddings'/run/'pp1.pkl' for name, run in MODELS.items()}
SOURCE_CONFIGS = {name: ROOT/'results'/run/'run_config.json' for name, run in MODELS.items()}
PAPER_REFERENCE = ROOT/'references/paper_figure2b_dreamwalk.csv'
for name in MODELS:
    if not EMBEDDINGS[name].is_file():
        raise FileNotFoundError(EMBEDDINGS[name])
    source = json.loads(SOURCE_CONFIGS[name].read_text())
    assert source['semantic'] == {'mesh_year': 2024, 'cutoff': 0.4, 'letter_roots': True}
    assert source['xgboost'] == XGB_PARAMETERS
    print(name, 'embedding:', EMBEDDINGS[name])
print('Repeat seeds:', REPEAT_SEEDS)
display(pd.DataFrame([XGB_PARAMETERS]))


original embedding: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/pp1.pkl
modified embedding: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_mesh2024_final_fixed/pp1.pkl
Repeat seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


,base_score,booster,eval_metric,objective,gamma,learning_rate,max_depth,n_estimators,tree_method,min_child_weight,subsample,colsample_bytree,scale_pos_weight,max_delta_step,seed,n_jobs
0,0.5,gbtree,error,binary:logistic,0,0.1,6,500,auto,4,0.8,0.9,1,1,0,4


## 3. Evaluation cohort and excluded labels

The accepted inputs contain 840 disease nodes and 5,926 positives. The category-based evaluation uses 748 diseases and 5,596 positives. Of those diseases, 398 have more than one highest-level category. Category counts below overlap because a disease can belong to multiple categories.

Category codes are the first full tree component (`C04`, not the letter `C`). The mapping and hierarchy are read from the completed preparation workflow.


In [15]:
cohort = area.load_cohort(ROOT)
assert len(cohort['drugs']) == 1661
assert len(cohort['categories']) == 748
assert len(cohort['positives']) == 5596
assert len(cohort['excluded_positives']) == 330
cohort_table = pd.DataFrame(cohort['audit'])
excluded = cohort_table.loc[~cohort_table['in_evaluation']].copy()
print('Eligible diseases:', len(cohort['categories']))
print('Eligible positives:', len(cohort['positives']))
print('Excluded diseases:', len(excluded))
print('Excluded positives:', len(cohort['excluded_positives']))
display(cohort_table.groupby(['in_evaluation', 'mapping_status'], dropna=False)
        .agg(diseases=('disease', 'size'), positives=('positive_pairs', 'sum')).reset_index())
category_counts = []
for category in sorted({c for values in cohort['categories'].values() for c in values}):
    members = {d for d, values in cohort['categories'].items() if category in values}
    category_counts.append({'category': category, 'diseases': len(members),
                            'positives': sum(d in members for _, d in cohort['positives'])})
display(pd.DataFrame(category_counts))
display(excluded)


Eligible diseases: 748
Eligible positives: 5596
Excluded diseases: 92
Excluded positives: 330


,in_evaluation,mapping_status,diseases,positives
0,False,NO_MSH_MAPPING,74,288
1,False,SUPPLEMENTARY_UNRESOLVED,18,42
2,True,DIRECT_UNIQUE_USABLE,748,5596


,category,diseases,positives
0,A10,1,3
1,C01,34,225
2,C04,127,761
3,C05,36,395
4,C06,58,411
5,C07,14,38
6,C08,44,522
7,C09,9,212
8,C10,120,680
9,C11,33,212


,disease,mesh_descriptor_id,mapping_status,categories,category_count,in_evaluation,positive_pairs
145,C0013144,,NO_MSH_MAPPING,,0,False,3
286,C0020640,,NO_MSH_MAPPING,,0,False,1
312,C0022596,,NO_MSH_MAPPING,,0,False,1
313,C0022638,,NO_MSH_MAPPING,,0,False,1
329,C0023449,,NO_MSH_MAPPING,,0,False,21
...,...,...,...,...,...,...,...
823,C2242472,,NO_MSH_MAPPING,,0,False,13
826,C2316810,,NO_MSH_MAPPING,,0,False,1
828,C2718128,,NO_MSH_MAPPING,,0,False,1
829,C2748055,,NO_MSH_MAPPING,,0,False,5


## 4. Generate and save ten shared splits and negative sets

Allocate the 30 categories as 24 training, 3 validation, and 3 test categories per repeat. Apply the test-first disease rule across all memberships. Thus, disease and pair counts can differ substantially from 8:1:1.

Each repeat samples 5,596 negatives uniformly from eligible unknown pairs. Negatives are unique within and across repeats, and never include any known positive. The same saved pairs feed both model variants. Seeds are fixed in advance; the notebook does not search for favorable splits.

Scientific data and settings identify a saved run. Editing notebook comments or Markdown does not change its identity. Change `RUN_NAME` when changing the experiment.


In [16]:
repeats = {}
used_negatives = set()
split_summary = []
for seed in REPEAT_SEEDS:
    category_split, disease_split, pairs, negatives, summary = area.make_repeat(cohort, seed, used_negatives)
    assert not used_negatives.intersection(negatives)
    used_negatives.update(negatives)
    repeats[seed] = {'categories': category_split, 'diseases': disease_split, 'pairs': pairs}
    split_summary.extend(summary)
assert len(used_negatives) == len(REPEAT_SEEDS) * len(cohort['positives'])
split_table = pd.DataFrame(split_summary)
display(split_table)


,repeat_seed,partition,allocated_categories,diseases,positives,negatives,pairs,positive_fraction
0,0,train,24,443,3919,3376,7295,0.537217
1,0,validation,3,196,961,1423,2384,0.403104
2,0,test,3,109,716,797,1513,0.473232
3,1,train,24,423,3210,3210,6420,0.500000
4,1,validation,3,139,934,991,1925,0.485195
5,1,test,3,186,1452,1395,2847,0.510011
6,2,train,24,497,3817,3676,7493,0.509409
7,2,validation,3,215,1550,1634,3184,0.486809
8,2,test,3,36,229,286,515,0.444660
9,3,train,24,436,2746,3265,6011,0.456829


In [7]:
input_paths = {
    'nodetypes': ROOT/'data/processed/msi_nodetypes.tsv',
    'positive_pairs': ROOT/'data/processed/msi_positive_pairs.tsv',
    'mesh_mapping': ROOT/'results/msi_mesh2024_demo_roots_inputs/mapping_audit.csv',
    'paper_reference': PAPER_REFERENCE,
    **{name+'_embedding': path for name, path in EMBEDDINGS.items()},
    **{name+'_source_config': path for name, path in SOURCE_CONFIGS.items()},
}
config = {
    'protocol_version': 1,
    'analysis': 'MSI Figure 2b reconstruction; paired existing original/modified seed-0 embeddings',
    'repeat_seeds': REPEAT_SEEDS,
    'category_ratio': [8, 1, 1],
    'category_policy': 'all accepted MeSH2024 tree roots; test priority, then validation, then train',
    'unmapped_policy': 'exclude labels from classifier evaluation; preserve graph nodes and embeddings',
    'negative_sampling': 'global 1:1; uniform unknown pairs; no cross-repeat overlap; partition by disease',
    'validation_policy': 'report only; no early stopping or parameter selection',
    'embedding_seed': 0, 'tau': 0.3, 'pp': 1, 'mesh_year': 2024, 'semantic_cutoff': 0.4,
    'feature': 'drug_embedding - disease_embedding',
    'accuracy_threshold': 0.5, 'aupr': 'average_precision_score',
    'xgboost': XGB_PARAMETERS,
    'cohort': {'drugs': len(cohort['drugs']), 'diseases': len(cohort['categories']),
               'positives': len(cohort['positives']), 'excluded_diseases': len(excluded),
               'excluded_positives': len(cohort['excluded_positives'])},
    'inputs': {name: {'path': str(path.relative_to(ROOT)), 'sha256': sha256(path)}
               for name, path in input_paths.items()},
    'splits': {str(seed): hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()
               for seed, value in repeats.items()},
    'python': sys.version,
    'packages': {name: version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'xgboost']},
}
area.open_run(RESULTS, config)
area.write_rows(RESULTS/'disease_categories.csv', cohort['audit'], list(cohort['audit'][0]))
area.write_rows(RESULTS/'excluded_diseases.csv', excluded.to_dict('records'), list(excluded.columns))
area.write_rows(RESULTS/'excluded_positive_pairs.csv',
                [{'drug': d, 'disease': s, 'label': 1} for d, s in cohort['excluded_positives']],
                ['drug', 'disease', 'label'])
for seed, value in repeats.items():
    area.save_repeat(RESULTS, seed, cohort['categories'], value['categories'], value['diseases'], value['pairs'])
area.write_rows(RESULTS/'split_summary.csv', split_summary, list(split_summary[0]))
print('Saved shared preparation:', RESULTS)


Saved shared preparation: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_disease_area_seed0_9


## 5. Load the two completed embeddings

Both models use the same required drug and disease IDs. Pair features remain the original 128D drug-minus-disease vectors. No scaler is fitted.


In [17]:
required_nodes = set(cohort['drugs']) | set(cohort['categories'])
original_vectors = area.load_vectors(EMBEDDINGS['original'], required_nodes)
modified_vectors = area.load_vectors(EMBEDDINGS['modified'], required_nodes)
print('Original embedding nodes:', len(original_vectors))
print('Modified embedding nodes:', len(modified_vectors))
example_pair = repeats[REPEAT_SEEDS[0]]['pairs'][0]
x_example, y_example = area.features(original_vectors, [example_pair])
print('Example:', example_pair)
print('Feature shape:', x_example.shape)


Original embedding nodes: 29959
Modified embedding nodes: 29959
Example: {'drug': '5-hydroxytryptophan', 'disease': 'C0917801', 'label': 1, 'partition': 'test'}
Feature shape: (1, 128)


## 6. Fit the original variant on each training partition

This cell starts classifier training. Each fit uses training pairs only, with the fixed 500-tree XGBoost settings. It records validation and test predictions, then saves `classifier.json`, `predictions.csv`, and `metrics.json` before proceeding. Re-running this cell reuses completed fits.


In [9]:
original_rows = []
for seed in REPEAT_SEEDS:
    print(f'Original variant: repeat {seed}', flush=True)
    rows = area.fit_repeat(original_vectors, repeats[seed]['pairs'],
                           RESULTS/'original'/f'repeat{seed:02d}', 'original', seed, XGB_PARAMETERS)
    original_rows.extend(rows)
    display(pd.DataFrame(rows))


Original variant: repeat 0


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,original,0,validation,961,1423,0.638423,0.675689,0.552693
1,original,0,test,716,797,0.618638,0.668367,0.607766


Original variant: repeat 1


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,original,1,validation,934,991,0.615584,0.659566,0.592692
1,original,1,test,1452,1395,0.588690,0.628736,0.637041


Original variant: repeat 2


...

,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,original,7,validation,488,756,0.605305,0.635062,0.536770
1,original,7,test,1260,1582,0.600985,0.623057,0.550287


Original variant: repeat 8


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,original,8,validation,721,684,0.592883,0.642068,0.651217
1,original,8,test,1455,1303,0.621827,0.689956,0.698210


Original variant: repeat 9


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,original,9,validation,1045,1212,0.622508,0.681212,0.616943
1,original,9,test,1553,1193,0.671886,0.748046,0.788270


## 7. Fit the modified variant on the identical partitions

The embedding vectors change. Pair IDs, labels, category assignments, classifier settings, and training seed are shared with Section 6.


In [10]:
modified_rows = []
for seed in REPEAT_SEEDS:
    print(f'Modified variant: repeat {seed}', flush=True)
    rows = area.fit_repeat(modified_vectors, repeats[seed]['pairs'],
                           RESULTS/'modified'/f'repeat{seed:02d}', 'modified', seed, XGB_PARAMETERS)
    modified_rows.extend(rows)
    display(pd.DataFrame(rows))


Modified variant: repeat 0


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,modified,0,validation,961,1423,0.658138,0.700626,0.621935
1,modified,0,test,716,797,0.686054,0.758369,0.717238


Modified variant: repeat 1


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,modified,1,validation,934,991,0.675325,0.757524,0.734025
1,modified,1,test,1452,1395,0.594310,0.691318,0.691397


Modified variant: repeat 2


...

,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,modified,7,validation,488,756,0.623794,0.641906,0.514395
1,modified,7,test,1260,1582,0.624208,0.657529,0.616233


Modified variant: repeat 8


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,modified,8,validation,721,684,0.599288,0.665965,0.681530
1,modified,8,test,1455,1303,0.637781,0.718199,0.729481


Modified variant: repeat 9


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,modified,9,validation,1045,1212,0.643332,0.727933,0.680758
1,modified,9,test,1553,1193,0.632921,0.744036,0.786828


## 8. Paired results across ten repeats

Report repeat means and sample SD (`ddof=1`). The main local comparison is **modified minus original on the same test pairs**. Validation results are shown separately. These are ten category-split repeats, not ten-fold CV.


In [11]:
all_rows = original_rows + modified_rows
assert len(all_rows) == 4 * len(REPEAT_SEEDS)
summary = area.summarize(all_rows)
deltas, delta_summary = area.paired_deltas(all_rows)
area.write_rows(RESULTS/'repeat_results.csv', all_rows, list(all_rows[0]))
area.write_rows(RESULTS/'summary.csv', summary, list(summary[0]))
area.write_rows(RESULTS/'paired_test_deltas.csv', deltas, list(deltas[0]))
area.write_rows(RESULTS/'paired_test_delta_summary.csv', delta_summary, list(delta_summary[0]))
summary_table = pd.DataFrame(summary)
print('Test summary:')
display(summary_table.loc[summary_table['partition'] == 'test'])
print('Validation summary:')
display(summary_table.loc[summary_table['partition'] == 'validation'])
print('Paired test differences, modified minus original:')
display(pd.DataFrame(deltas))
display(pd.DataFrame(delta_summary))


Test summary:


,model,partition,metric,mean,SD,repeats
3,modified,test,accuracy,0.629715,0.027745,10
4,modified,test,auroc,0.711832,0.030849,10
5,modified,test,aupr,0.702729,0.066681,10
9,original,test,accuracy,0.609007,0.037198,10
10,original,test,auroc,0.671951,0.036579,10
11,original,test,aupr,0.662569,0.068683,10


Validation summary:


,model,partition,metric,mean,SD,repeats
0,modified,validation,accuracy,0.628692,0.041781,10
1,modified,validation,auroc,0.694542,0.041035,10
2,modified,validation,aupr,0.658884,0.059983,10
6,original,validation,accuracy,0.625415,0.017784,10
7,original,validation,auroc,0.676613,0.030425,10
8,original,validation,aupr,0.631438,0.069321,10


Paired test differences, modified minus original:


,repeat_seed,accuracy,auroc,aupr
0,0,0.067416,0.090002,0.109472
1,1,0.005620,0.062582,0.054356
2,2,0.038835,0.002122,-0.017890
3,3,0.010694,0.040405,0.050086
4,4,0.098846,0.087028,0.057715
5,5,-0.001789,0.017697,0.012513
6,6,-0.012748,0.040267,0.039569
7,7,0.023223,0.034472,0.065946
8,8,0.015954,0.028244,0.031271
9,9,-0.038966,-0.004010,-0.001443


,metric,mean_modified_minus_original,SD,repeats
0,accuracy,0.020708,0.039664,10
1,auroc,0.039881,0.032076,10
2,aupr,0.040160,0.036548,10


## 9. Compare with the paper's Figure 2b

The reference CSV contains the ten DREAMwalk rows from the authors' Source Data workbook, sheet `Figure2b`. Its mean AUROC is approximately 0.820409; do not use the Figure 2a AUROC of 0.944060 as the disease-area target.

Paper values are context for reproduction. The original category assignments and negative-pair IDs are unavailable. This local cohort excludes unclassified diseases and applies the approved all-membership policy. The paired local comparison is therefore more directly controlled than either local-to-paper comparison.


In [12]:
paper = pd.read_csv(PAPER_REFERENCE)
assert len(paper) == 10
paper_summary = {metric: {'mean': float(paper[metric].mean()), 'SD': float(paper[metric].std(ddof=1))}
                 for metric in area.METRICS}
comparison = []
for row in summary:
    if row['partition'] == 'test':
        reference = paper_summary[row['metric']]
        comparison.append({'model': row['model'], 'metric': row['metric'],
                           'local_mean': row['mean'], 'local_repeat_SD': row['SD'],
                           'paper_mean': reference['mean'], 'paper_repeat_SD': reference['SD'],
                           'local_minus_paper': row['mean'] - reference['mean']})
area.write_rows(RESULTS/'paper_comparison.csv', comparison, list(comparison[0]))
display(pd.DataFrame(comparison))
area.write_json(RESULTS/'evaluation_complete.json',
                {'models': list(MODELS), 'repeat_seeds': REPEAT_SEEDS,
                 'completed_classifier_fits': 2 * len(REPEAT_SEEDS),
                 'uncertainty': 'split/negative-set variation conditional on fixed seed-0 embeddings'})
print('Completed evaluation:', RESULTS)


,model,metric,local_mean,local_repeat_SD,paper_mean,paper_repeat_SD,local_minus_paper
0,modified,accuracy,0.629715,0.027745,0.692152,0.035304,-0.062437
1,modified,auroc,0.711832,0.030849,0.820409,0.033920,-0.108578
2,modified,aupr,0.702729,0.066681,0.783385,0.038025,-0.080656
3,original,accuracy,0.609007,0.037198,0.692152,0.035304,-0.083145
4,original,auroc,0.671951,0.036579,0.820409,0.033920,-0.148459
5,original,aupr,0.662569,0.068683,0.783385,0.038025,-0.120816


Completed evaluation: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_disease_area_seed0_9


## 10. Interpretation and saved outputs

- Test-category disease labels never enter classifier training or validation. Validation-category labels never enter training.
- With multiple MeSH memberships, test diseases may also carry training-category annotations. The priority rule removes those entire diseases from training; it does not pretend that each disease has only one biological category.
- Globally balanced negatives do not imply balanced labels in each disease partition. Read `split_summary.csv` alongside performance.
- All nodes already have embeddings. This evaluation does not predict embeddings for nodes absent from MSI.
- SD excludes new walk/embedding runs. Report this scope with the results.

Outputs under `results/<RUN_NAME>/` include the cohort/exclusion audit, shared category/disease/pair splits, per-fit classifiers and predictions, repeat metrics, paired differences, and the paper comparison. Training is started by the user through Sections 6 and 7.


## 11. Relaxed control: one fixed evaluation category per disease

The strict rule assigns an entire disease to test if **any** accepted category belongs to test, then applies validation priority. For example, a disease annotated with both training category `C04` and test category `C10` enters test.

This control chooses one accepted category per disease **once**, using assignment seed 0, before examining any labels or performance. Diseases and their category lists are sorted before sampling, so input row order does not affect the choice. The chosen category is held fixed across all ten repeats and both model variants. Here, "primary" is an evaluation assignment, not an official MeSH primary designation.

The same example enters training if `C04` is selected, or test if `C10` is selected. Every disease still belongs to exactly one partition. Different diseases can share secondary categories across partitions, so this control does not enforce isolation across every MeSH membership. It does not duplicate a disease or its labels between training and test.

Keep Sections 1-5 as shared preparation. The strict result directory remains separate. Model settings, embeddings, excluded diseases, category allocations and negative-pair IDs stay the same; only the rule assigning diseases to partitions changes.


In [18]:
import importlib
area = importlib.reload(area)

PRIMARY_ASSIGNMENT_SEED = 0
PRIMARY_RUN_NAME = 'msi_mesh2024_disease_area_primary_seed0_9'
PRIMARY_RESULTS = ROOT/'results'/PRIMARY_RUN_NAME
assert PRIMARY_RESULTS != RESULTS
primary_categories = area.select_primary_categories(cohort['categories'], PRIMARY_ASSIGNMENT_SEED)
primary_audit = [dict(row, primary_category=primary_categories.get(row['disease'], ''))
                 for row in cohort['audit']]
print('Fixed assignment seed:', PRIMARY_ASSIGNMENT_SEED)
print('New outputs:', PRIMARY_RESULTS)
display(pd.DataFrame(primary_audit).loc[lambda table: table['in_evaluation'],
        ['disease', 'categories', 'primary_category', 'positive_pairs']].head(20))


Fixed assignment seed: 0
New outputs: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_disease_area_primary_seed0_9


,disease,categories,primary_category,positive_pairs
0,C0000737,C23,C23,1
1,C0000744,C16;C18,C18,1
2,C0000786,C12;G08,C12,3
3,C0000809,C12,C12,1
4,C0001122,C18,C18,3
5,C0001126,C12;C16;C18,C16,1
6,C0001206,C05;C10;C19,C10,7
7,C0001403,C19;C20,C20,11
8,C0001418,C04,C04,1
9,C0001430,C04,C04,1


## 12. Prepare primary-category splits and check the control

Allocate the same 30 categories to 24 training, 3 validation and 3 test categories using repeat seeds 0-9. Each disease follows its fixed selected category. The primary assignment is not redrawn for each repeat.

The assertions below check that category allocations and ordered `(drug, disease, label)` rows match the strict control exactly. Negatives remain unique across repeats. Only partition assignments may differ. Counts are not forced to 8:1:1 at the disease or pair level.

`diseases_with_test_category` shows actual test-category memberships in each partition, including secondary categories. Under this relaxed rule, its training and validation counts can exceed zero. All labels of an individual disease still stay together.


In [19]:
primary_repeats = {}
primary_used_negatives = set()
primary_split_rows = []
for seed in REPEAT_SEEDS:
    allocated, disease_split, pairs, negatives, rows = area.make_repeat(
        cohort, seed, primary_used_negatives, primary_categories)
    assert allocated == repeats[seed]['categories']
    assert [(r['drug'], r['disease'], r['label']) for r in pairs] == [
        (r['drug'], r['disease'], r['label']) for r in repeats[seed]['pairs']]
    assert not primary_used_negatives.intersection(negatives)
    primary_used_negatives.update(negatives)
    for row in rows:
        row['diseases_with_test_category'] = sum(
            disease_split[disease] == row['partition'] and
            any(allocated[category] == 'test' for category in categories)
            for disease, categories in cohort['categories'].items())
    primary_repeats[seed] = {'categories': allocated, 'diseases': disease_split, 'pairs': pairs}
    primary_split_rows.extend(rows)
assert len(primary_used_negatives) == len(REPEAT_SEEDS) * len(cohort['positives'])
primary_split_table = pd.DataFrame(primary_split_rows)
print('Primary-category split counts:')
display(primary_split_table)
print('Training sizes under each policy:')
display(pd.concat([
    split_table.loc[split_table['partition'] == 'train'].assign(split_policy='all_memberships'),
    primary_split_table.loc[primary_split_table['partition'] == 'train'].assign(split_policy='fixed_primary'),
], ignore_index=True)[['split_policy', 'repeat_seed', 'diseases', 'positives', 'negatives', 'pairs']]
    .sort_values(['repeat_seed', 'split_policy']))


Primary-category split counts:


,repeat_seed,partition,allocated_categories,diseases,positives,negatives,pairs,positive_fraction,diseases_with_test_category
0,0,train,24,552,4564,4127,8691,0.525141,26
1,0,validation,3,125,660,924,1584,0.416667,12
2,0,test,3,71,372,545,917,0.405671,71
3,1,train,24,538,4049,4057,8106,0.499507,71
4,1,validation,3,102,673,717,1390,0.484173,7
5,1,test,3,108,874,822,1696,0.515330,108
6,2,train,24,591,4406,4382,8788,0.501365,14
7,2,validation,3,138,1097,1069,2166,0.506464,3
8,2,test,3,19,93,145,238,0.390756,19
9,3,train,24,551,3714,4128,7842,0.473604,81


Training sizes under each policy:


,split_policy,repeat_seed,diseases,positives,negatives,pairs
0,all_memberships,0,443,3919,3376,7295
10,fixed_primary,0,552,4564,4127,8691
1,all_memberships,1,423,3210,3210,6420
11,fixed_primary,1,538,4049,4057,8106
2,all_memberships,2,497,3817,3676,7493
12,fixed_primary,2,591,4406,4382,8788
3,all_memberships,3,436,2746,3265,6011
13,fixed_primary,3,551,3714,4128,7842
4,all_memberships,4,515,3850,3861,7711
14,fixed_primary,4,603,4498,4531,9029


In [20]:
primary_config = dict(
    config,
    protocol_version=2,
    analysis='MSI disease-area sensitivity; fixed primary categories; existing original/modified seed-0 embeddings',
    category_policy='one fixed uniformly selected accepted category per disease; all category allocations unchanged',
    primary_assignment_seed=PRIMARY_ASSIGNMENT_SEED,
    primary_categories=primary_categories,
    splits={str(seed): hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()
            for seed, value in primary_repeats.items()},
)
area.open_run(PRIMARY_RESULTS, primary_config)
area.write_rows(PRIMARY_RESULTS/'disease_categories.csv', primary_audit, list(primary_audit[0]))
area.write_rows(PRIMARY_RESULTS/'excluded_diseases.csv', excluded.to_dict('records'), list(excluded.columns))
area.write_rows(PRIMARY_RESULTS/'excluded_positive_pairs.csv',
                [{'drug': d, 'disease': s, 'label': 1} for d, s in cohort['excluded_positives']],
                ['drug', 'disease', 'label'])
for seed, value in primary_repeats.items():
    area.save_repeat(PRIMARY_RESULTS, seed, cohort['categories'],
                     value['categories'], value['diseases'], value['pairs'])
area.write_rows(PRIMARY_RESULTS/'split_summary.csv', primary_split_rows, list(primary_split_rows[0]))
print('Saved primary-category preparation:', PRIMARY_RESULTS)


Saved primary-category preparation: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_disease_area_primary_seed0_9


## 13. Primary-category control: fit the original variant

This cell starts ten new classifier fits using the original embeddings. It reuses the existing 500-tree XGBoost settings and classifier seed 0. Each completed fit is saved immediately and reused on a subsequent run.


In [21]:
primary_original_rows = []
for seed in REPEAT_SEEDS:
    print(f'Primary-category original: repeat {seed}', flush=True)
    rows = area.fit_repeat(original_vectors, primary_repeats[seed]['pairs'],
                           PRIMARY_RESULTS/'original'/f'repeat{seed:02d}',
                           'original', seed, XGB_PARAMETERS)
    primary_original_rows.extend(rows)
    display(pd.DataFrame(rows))


Primary-category original: repeat 0


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,original,0,validation,660,924,0.657828,0.702714,0.620911
1,original,0,test,372,545,0.668484,0.697830,0.546239


Primary-category original: repeat 1


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,original,1,validation,673,717,0.666906,0.713991,0.655183
1,original,1,test,874,822,0.598467,0.660193,0.658586


Primary-category original: repeat 2


...

,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,original,7,validation,572,737,0.634836,0.672702,0.596315
1,original,7,test,761,956,0.619685,0.651114,0.580867


Primary-category original: repeat 8


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,original,8,validation,663,547,0.612397,0.677222,0.715058
1,original,8,test,805,809,0.667286,0.753980,0.742692


Primary-category original: repeat 9


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,original,9,validation,820,817,0.643250,0.714534,0.705135
1,original,9,test,1080,810,0.669312,0.748721,0.799084


## 14. Primary-category control: fit the modified variant

Use the identical primary-category splits, pair IDs and classifier settings from Section 13. Only the existing embedding vectors change to the combined modified variant.


In [22]:
primary_modified_rows = []
for seed in REPEAT_SEEDS:
    print(f'Primary-category modified: repeat {seed}', flush=True)
    rows = area.fit_repeat(modified_vectors, primary_repeats[seed]['pairs'],
                           PRIMARY_RESULTS/'modified'/f'repeat{seed:02d}',
                           'modified', seed, XGB_PARAMETERS)
    primary_modified_rows.extend(rows)
    display(pd.DataFrame(rows))


Primary-category modified: repeat 0


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,modified,0,validation,660,924,0.710227,0.772991,0.691310
1,modified,0,test,372,545,0.703381,0.787403,0.661143


Primary-category modified: repeat 1


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,modified,1,validation,673,717,0.734532,0.821174,0.787190
1,modified,1,test,874,822,0.610849,0.693877,0.709712


Primary-category modified: repeat 2


...

,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,modified,7,validation,572,737,0.689076,0.763599,0.703662
1,modified,7,test,761,956,0.622598,0.666637,0.608748


Primary-category modified: repeat 8


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,modified,8,validation,663,547,0.657851,0.729563,0.785252
1,modified,8,test,805,809,0.630112,0.738925,0.710069


Primary-category modified: repeat 9


,model,repeat_seed,partition,positives,negatives,accuracy,auroc,aupr
0,modified,9,validation,820,817,0.642028,0.752257,0.746302
1,modified,9,test,1080,810,0.661905,0.765979,0.809665


## 15. Compare original and modified models, both split policies, and the paper

Original-versus-modified differences within the primary-category policy use identical test pairs. Strict-versus-primary comparisons use different test cohorts and are descriptive, not paired comparisons on the same test pairs. A higher score under one policy does not establish the authors' unpublished overlap rule.


In [23]:
primary_all_rows = primary_original_rows + primary_modified_rows
assert len(primary_all_rows) == 4 * len(REPEAT_SEEDS)
primary_summary = area.summarize(primary_all_rows)
primary_deltas, primary_delta_summary = area.paired_deltas(primary_all_rows)
area.write_rows(PRIMARY_RESULTS/'repeat_results.csv', primary_all_rows, list(primary_all_rows[0]))
area.write_rows(PRIMARY_RESULTS/'summary.csv', primary_summary, list(primary_summary[0]))
area.write_rows(PRIMARY_RESULTS/'paired_test_deltas.csv', primary_deltas, list(primary_deltas[0]))
area.write_rows(PRIMARY_RESULTS/'paired_test_delta_summary.csv',
                primary_delta_summary, list(primary_delta_summary[0]))
primary_summary_table = pd.DataFrame(primary_summary)
print('Primary-category test summary:')
display(primary_summary_table.loc[primary_summary_table['partition'] == 'test'])
print('Paired modified-minus-original differences within primary-category splits:')
display(pd.DataFrame(primary_delta_summary))

paper_reference = pd.read_csv(PAPER_REFERENCE)
assert len(paper_reference) == 10
primary_paper_comparison = []
for row in primary_summary:
    if row['partition'] == 'test':
        primary_paper_comparison.append(dict(
            model=row['model'], metric=row['metric'], local_mean=row['mean'],
            local_repeat_SD=row['SD'], paper_mean=float(paper_reference[row['metric']].mean()),
            paper_repeat_SD=float(paper_reference[row['metric']].std(ddof=1)),
            local_minus_paper=row['mean'] - float(paper_reference[row['metric']].mean())))
area.write_rows(PRIMARY_RESULTS/'paper_comparison.csv',
                primary_paper_comparison, list(primary_paper_comparison[0]))

policy_tables = [primary_summary_table.assign(split_policy='fixed_primary')]
if (RESULTS/'summary.csv').is_file():
    policy_tables.append(pd.read_csv(RESULTS/'summary.csv').assign(split_policy='all_memberships'))
else:
    print('Strict model fits are not complete; their summary is omitted.')
policy_tables.append(pd.DataFrame([
    dict(split_policy='paper_unpublished_overlap_rule', model='DREAMwalk', partition='test',
         metric=metric, mean=float(paper_reference[metric].mean()),
         SD=float(paper_reference[metric].std(ddof=1)), repeats=len(paper_reference))
    for metric in area.METRICS]))
policy_comparison = pd.concat(policy_tables, ignore_index=True)
policy_comparison = policy_comparison.loc[policy_comparison['partition'] == 'test',
    ['split_policy', 'model', 'metric', 'mean', 'SD', 'repeats']]
area.write_rows(PRIMARY_RESULTS/'split_policy_comparison.csv',
                policy_comparison.to_dict('records'), list(policy_comparison.columns))
print('Both split policies and the paper:')
display(policy_comparison.sort_values(['metric', 'split_policy', 'model']))
area.write_json(PRIMARY_RESULTS/'evaluation_complete.json', dict(
    models=list(MODELS), repeat_seeds=REPEAT_SEEDS,
    primary_assignment_seed=PRIMARY_ASSIGNMENT_SEED,
    completed_classifier_fits=2 * len(REPEAT_SEEDS),
    uncertainty='split/negative-set variation conditional on fixed embeddings and primary assignments'))
print('Completed primary-category evaluation:', PRIMARY_RESULTS)


Primary-category test summary:


,model,partition,metric,mean,SD,repeats
3,modified,test,accuracy,0.646006,0.049306,10
4,modified,test,auroc,0.738424,0.040381,10
5,modified,test,aupr,0.720276,0.069725,10
9,original,test,accuracy,0.639260,0.054186,10
10,original,test,auroc,0.702606,0.038208,10
11,original,test,aupr,0.688497,0.084745,10


Paired modified-minus-original differences within primary-category splits:


,metric,mean_modified_minus_original,SD,repeats
0,accuracy,0.006745,0.035906,10
1,auroc,0.035817,0.027695,10
2,aupr,0.031779,0.038709,10


Both split policies and the paper:


,split_policy,model,metric,mean,SD,repeats
15,all_memberships,modified,accuracy,0.629715,0.027745,10
21,all_memberships,original,accuracy,0.609007,0.037198,10
3,fixed_primary,modified,accuracy,0.646006,0.049306,10
9,fixed_primary,original,accuracy,0.639260,0.054186,10
24,paper_unpublished_overlap_rule,DREAMwalk,accuracy,0.692152,0.035304,10
17,all_memberships,modified,aupr,0.702729,0.066681,10
23,all_memberships,original,aupr,0.662569,0.068683,10
5,fixed_primary,modified,aupr,0.720276,0.069725,10
11,fixed_primary,original,aupr,0.688497,0.084745,10
26,paper_unpublished_overlap_rule,DREAMwalk,aupr,0.783385,0.038025,10


Completed primary-category evaluation: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_disease_area_primary_seed0_9


## 16. Interpret the relaxed control

- Strict all-membership isolation and fixed-primary assignment answer different category-generalization questions. Every disease's classifier labels remain disjoint between training, validation and test in both policies.
- The primary policy allows secondary-category overlap across partitions. Read training/validation `diseases_with_test_category` alongside the performance table.
- The primary assignment seed is fixed at 0 before fitting. Do not search assignment seeds or select a mapping using test performance.
- Embeddings, category allocations, positive IDs, negative IDs, cohort exclusions and classifier settings are shared with the strict control. All differences in pair partitions follow from the category-assignment policy.
- The paper did not publish its multi-category assignment rule. This is a documented sensitivity analysis, not confirmation of the exact Figure 2b split.
- Existing strict scientific outputs are retained under `results/msi_mesh2024_disease_area_seed0_9/`. New outputs are saved under `results/msi_mesh2024_disease_area_primary_seed0_9/`, including the fixed assignment audit, per-repeat splits and checkpoints, paired model differences, and `split_policy_comparison.csv`.
